# Data Merging

## Objective

The objective of this notebook is to combine all cleaned datasets into a single master dataset for Exploratory Data Analysis (EDA), SQL analysis, Power BI dashboard creation, and Machine Learning.

During this notebook, we will:

- Load the cleaned datasets.
- Understand relationships between datasets.
- Merge datasets step by step.
- Verify each merge.
- Create a final master dataset.
- Save the merged dataset for future analysis.

In [2]:
import pandas as pd
import numpy as np

In [3]:
print("=" * 90)
print("LOADING CLEANED DATASETS")
print("=" * 90)

LOADING CLEANED DATASETS


In [4]:
customers = pd.read_csv("../data/processed/customers_cleaned.csv")

orders = pd.read_csv("../data/processed/orders_cleaned.csv")

order_items = pd.read_csv("../data/processed/order_items_cleaned.csv")

payments = pd.read_csv("../data/processed/payments_cleaned.csv")

products = pd.read_csv("../data/processed/products_cleaned.csv")

reviews = pd.read_csv("../data/processed/reviews_cleaned.csv")

In [5]:
print("=" * 90)
print("DATASET SHAPES")
print("=" * 90)

print("Customers :", customers.shape)
print("Orders :", orders.shape)
print("Order Items :", order_items.shape)
print("Payments :", payments.shape)
print("Products :", products.shape)
print("Reviews :", reviews.shape)

DATASET SHAPES
Customers : (99441, 5)
Orders : (99441, 8)
Order Items : (112650, 7)
Payments : (103886, 5)
Products : (32951, 9)
Reviews : (99224, 7)


# Dataset Relationships

Understanding relationships between datasets is essential before merging.

| Dataset | Primary Key | Foreign Key | Relationship |
|----------|-------------|-------------|--------------|
| Customers | customer_id | customer_id (Orders) | One-to-Many |
| Orders | order_id | order_id (Order Items) | One-to-Many |
| Orders | order_id | order_id (Payments) | One-to-Many |
| Orders | order_id | order_id (Reviews) | One-to-One (mostly) |
| Products | product_id | product_id (Order Items) | One-to-Many |

# Merge Strategy

The datasets will be merged in the following sequence:

1. Orders + Customers
2. Result + Order Items
3. Result + Products
4. Result + Payments
5. Result + Reviews

After every merge, the merged dataset will be verified by checking:

- Dataset shape
- Missing values
- Duplicate records
- Data consistency

This step-by-step approach helps identify issues early and ensures a reliable master dataset.

# Merge 1: Orders + Customers

The first merge combines the **Orders** dataset with the **Customers** dataset using the `customer_id` column.

### Why merge these datasets?

- Every order belongs to one customer.
- A customer can place multiple orders.
- This represents a **One-to-Many** relationship.
- We use a **Left Join** to preserve all order records while adding customer information.

In [9]:
print("=" * 90)
print("MERGE 1 : ORDERS + CUSTOMERS")
print("=" * 90)

MERGE 1 : ORDERS + CUSTOMERS


In [10]:
print("Orders Shape :", orders.shape)
print("Customers Shape :", customers.shape)

Orders Shape : (99441, 8)
Customers Shape : (99441, 5)


In [11]:
orders_customers = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="left"
)

In [15]:
print("=" * 90)
print("VERIFYING MERGE")
print("=" * 90)

print("Orders Shape :", orders.shape)
print("Customers Shape :", customers.shape)
print("Merged Shape :", orders_customers.shape)

VERIFYING MERGE
Orders Shape : (99441, 8)
Customers Shape : (99441, 5)
Merged Shape : (99441, 12)


In [16]:
orders_customers.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [17]:
print("=" * 90)
print("CHECKING DUPLICATE ORDER IDs")
print("=" * 90)

duplicate_orders = orders_customers["order_id"].duplicated().sum()

print("Duplicate Order IDs :", duplicate_orders)

CHECKING DUPLICATE ORDER IDs
Duplicate Order IDs : 0


In [18]:
print("=" * 90)
print("MISSING VALUES AFTER MERGE")
print("=" * 90)

orders_customers.isnull().sum()

MISSING VALUES AFTER MERGE


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
dtype: int64

## Observation

The Orders dataset was successfully merged with the Customers dataset using a Left Join on the `customer_id` column.

### Verification Results

- The number of rows remained unchanged.
- Customer information was successfully added to every order.
- No duplicate order IDs were created.
- Existing missing values belong to the Orders dataset and represent valid business scenarios.
- The merged dataset is ready for the next merge with the Order Items dataset.

# Merge 2: Orders + Customers + Order Items

The second merge combines the merged Orders-Customers dataset with the Order Items dataset using the `order_id` column.

### Why merge these datasets?

- Each order can contain one or more products.
- The Order Items dataset stores the products associated with each order.
- This represents a **One-to-Many** relationship.
- A Left Join is used to preserve all orders while adding product-level details.

In [21]:
print("=" * 90)
print("MERGE 2 : ORDERS + CUSTOMERS + ORDER ITEMS")
print("=" * 90)

MERGE 2 : ORDERS + CUSTOMERS + ORDER ITEMS


In [22]:
print("Orders + Customers Shape :", orders_customers.shape)
print("Order Items Shape :", order_items.shape)

Orders + Customers Shape : (99441, 12)
Order Items Shape : (112650, 7)


In [23]:
orders_customers_items = pd.merge(
    orders_customers,
    order_items,
    on="order_id",
    how="left"
)

In [24]:
print("=" * 90)
print("VERIFYING MERGE")
print("=" * 90)

print("Orders + Customers Shape :", orders_customers.shape)
print("Order Items Shape :", order_items.shape)
print("Merged Shape :", orders_customers_items.shape)

VERIFYING MERGE
Orders + Customers Shape : (99441, 12)
Order Items Shape : (112650, 7)
Merged Shape : (113425, 18)


In [25]:
print("=" * 90)
print("ORDERS WITHOUT ORDER ITEMS")
print("=" * 90)

missing_items = orders_customers_items["product_id"].isnull().sum()

print("Orders without Order Items:", missing_items)

ORDERS WITHOUT ORDER ITEMS
Orders without Order Items: 775


In [26]:
print("=" * 90)
print("UNIQUE ORDER IDS")
print("=" * 90)

print("Orders Dataset:", orders["order_id"].nunique())
print("Order Items Dataset:", order_items["order_id"].nunique())
print("Merged Dataset:", orders_customers_items["order_id"].nunique())

UNIQUE ORDER IDS
Orders Dataset: 99441
Order Items Dataset: 98666
Merged Dataset: 99441


In [27]:
orders_customers_items.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


In [28]:
print("=" * 90)
print("CHECKING DUPLICATE ORDER ITEMS")
print("=" * 90)

duplicates = orders_customers_items.duplicated(
    subset=["order_id", "order_item"]
).sum()

print("Duplicate Order Items :", duplicates)

CHECKING DUPLICATE ORDER ITEMS


KeyError: Index(['order_item'], dtype='object')

In [29]:
orders_customers_items.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

In [30]:
print("=" * 90)
print("CHECKING DUPLICATE ORDER ITEMS")
print("=" * 90)

duplicates = orders_customers_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

print("Duplicate Order Items:", duplicates)

CHECKING DUPLICATE ORDER ITEMS
Duplicate Order Items: 0


# Merge 3: Orders + Customers + Order Items + Products

The third merge combines the merged dataset with the Products dataset using the `product_id` column.

### Why merge these datasets?

- The Order Items dataset contains only product IDs.
- The Products dataset contains product details such as category, weight, dimensions, and description.
- This is a **Many-to-One** relationship because many order items can reference the same product.
- A Left Join is used to preserve all order-item records.

In [33]:
print("=" * 90)
print("MERGE 3 : ADDING PRODUCT DETAILS")
print("=" * 90)

MERGE 3 : ADDING PRODUCT DETAILS


In [34]:
print("Current Master Shape :", orders_customers_items.shape)
print("Products Shape :", products.shape)

Current Master Shape : (113425, 18)
Products Shape : (32951, 9)


In [35]:
orders_master = pd.merge(
    orders_customers_items,
    products,
    on="product_id",
    how="left"
)

In [36]:
print("=" * 90)
print("VERIFYING MERGE")
print("=" * 90)

print("Before Merge :", orders_customers_items.shape)
print("Products :", products.shape)
print("After Merge :", orders_master.shape)

VERIFYING MERGE
Before Merge : (113425, 18)
Products : (32951, 9)
After Merge : (113425, 26)


In [37]:
orders_master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0


In [38]:
print("=" * 90)
print("CHECKING DUPLICATES")
print("=" * 90)

duplicates = orders_master.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

print("Duplicate Order Items:", duplicates)

CHECKING DUPLICATES
Duplicate Order Items: 0


In [39]:
print("=" * 90)
print("CHECKING MISSING VALUES")
print("=" * 90)

orders_master.isnull().sum()

CHECKING MISSING VALUES


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 161
order_delivered_carrier_date     1968
order_delivered_customer_date    3229
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
order_item_id                     775
product_id                        775
seller_id                         775
shipping_limit_date               775
price                             775
freight_value                     775
product_category_name             775
product_name_lenght               775
product_description_lenght        775
product_photos_qty                775
product_weight_g                  775
product_length_cm                 775
product_height_cm                 775
product_width_cm                  775
dtype: int64

## Observation

The merged dataset was successfully combined with the Products dataset using the `product_id` column.

### Verification Results

- Rows before merge: **113,425**
- Rows after merge: **113,425**
- No rows were lost or duplicated.
- Product details such as category, dimensions, weight, and description were successfully added.
- The merged dataset now contains **26 columns** because the common merge key (`product_id`) appears only once.
- The dataset is ready for the next merge with the Payments dataset.

# Merge 4: Adding Payment Information

The fourth merge combines the master dataset with the Payments dataset using the `order_id` column.

### Why merge these datasets?

- Each order has one or more payment records.
- The Payments dataset contains payment method, installment count, and payment value.
- This represents a **One-to-Many** relationship because an order may have multiple payment transactions.
- A Left Join is used to preserve all order records.

In [50]:
print("=" * 90)
print("MERGE 4 : ADDING PAYMENT DETAILS")
print("=" * 90)

MERGE 4 : ADDING PAYMENT DETAILS


In [51]:
print("Master Dataset Shape :", orders_master.shape)
print("Payments Shape :", payments.shape)

Master Dataset Shape : (140564, 34)
Payments Shape : (103886, 5)


In [53]:
orders_master = pd.merge(
    orders_customers_items,
    products,
    on="product_id",
    how="left"
)

print(orders_master.shape)

(113425, 26)


In [54]:
before_shape = orders_master.shape

orders_master = pd.merge(
    orders_master,
    payments,
    on="order_id",
    how="left"
)

print("Before Merge :", before_shape)
print("Payments Shape :", payments.shape)
print("After Merge :", orders_master.shape)

Before Merge : (113425, 26)
Payments Shape : (103886, 5)
After Merge : (118434, 30)


In [55]:
[col for col in orders_master.columns if "payment" in col]

['payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

In [56]:
orders_master.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value']

In [57]:
'''
## Observation

The master dataset was successfully merged with the Payments dataset using the `order_id` column.

### Verification Results

- Payment information was successfully added.
- New columns added:
  - payment_sequential
  - payment_type
  - payment_installments
  - payment_value
- No duplicate payment columns were created.
- The dataset is now ready for the final merge with the Reviews dataset.
'''

'\n## Observation\n\nThe master dataset was successfully merged with the Payments dataset using the `order_id` column.\n\n### Verification Results\n\n- Payment information was successfully added.\n- New columns added:\n  - payment_sequential\n  - payment_type\n  - payment_installments\n  - payment_value\n- No duplicate payment columns were created.\n- The dataset is now ready for the final merge with the Reviews dataset.\n'

In [58]:
'''
# Merge 5: Adding Customer Reviews

The final merge combines the master dataset with the Reviews dataset using the `order_id` column.

### Why merge these datasets?

- The Reviews dataset contains customer ratings and feedback.
- Customer reviews help analyze customer satisfaction and service quality.
- Most orders have one review, making this generally a **One-to-One** relationship.
- A Left Join is used to preserve all order records.
'''

'\n# Merge 5: Adding Customer Reviews\n\nThe final merge combines the master dataset with the Reviews dataset using the `order_id` column.\n\n### Why merge these datasets?\n\n- The Reviews dataset contains customer ratings and feedback.\n- Customer reviews help analyze customer satisfaction and service quality.\n- Most orders have one review, making this generally a **One-to-One** relationship.\n- A Left Join is used to preserve all order records.\n'

In [59]:
print("=" * 90)
print("MERGE 5 : ADDING CUSTOMER REVIEWS")
print("=" * 90)

MERGE 5 : ADDING CUSTOMER REVIEWS


In [60]:
print("Master Dataset Shape :", orders_master.shape)
print("Reviews Shape :", reviews.shape)

Master Dataset Shape : (118434, 30)
Reviews Shape : (99224, 7)


In [61]:
before_shape = orders_master.shape

orders_master = pd.merge(
    orders_master,
    reviews,
    on="order_id",
    how="left"
)

In [62]:
print("=" * 90)
print("VERIFYING MERGE")
print("=" * 90)

print("Before Merge :", before_shape)
print("Reviews Shape :", reviews.shape)
print("After Merge :", orders_master.shape)

VERIFYING MERGE
Before Merge : (118434, 30)
Reviews Shape : (99224, 7)
After Merge : (119143, 36)


In [63]:
orders_master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,credit_card,1.0,18.12,a54f0611adc9ed256b57ede6b6eb5114,4.0,No Title,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,3.0,voucher,1.0,2.00,a54f0611adc9ed256b57ede6b6eb5114,4.0,No Title,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,2.0,voucher,1.0,18.59,a54f0611adc9ed256b57ede6b6eb5114,4.0,No Title,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,boleto,1.0,141.46,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,credit_card,3.0,179.12,e73b67b67587f7644d5bd1a52deb1b01,5.0,No Title,No Comment,2018-08-18 00:00:00,2018-08-22 19:07:58


In [71]:
print("=" * 90)
print("CHECKING DUPLICATES IN ORIGINAL REVIEWS DATASET")
print("=" * 90)

duplicates = reviews.duplicated(
    subset=["order_id", "review_id"]
).sum()

print("Duplicate Reviews in Original Dataset:", duplicates)

CHECKING DUPLICATES IN ORIGINAL REVIEWS DATASET
Duplicate Reviews in Original Dataset: 0


In [73]:
'''
## Observation

The duplicate check on the merged dataset showed repeated review records. After investigation, it was found that:

- The original Reviews dataset contains **0 duplicate records**.
- The repeated reviews in the master dataset are expected because the dataset is maintained at the **order-item level**.
- A single review is repeated for each product in the same order after merging.
- Therefore, these are **not data quality issues** and no rows were removed.
'''

'\n## Observation\n\nThe duplicate check on the merged dataset showed repeated review records. After investigation, it was found that:\n\n- The original Reviews dataset contains **0 duplicate records**.\n- The repeated reviews in the master dataset are expected because the dataset is maintained at the **order-item level**.\n- A single review is repeated for each product in the same order after merging.\n- Therefore, these are **not data quality issues** and no rows were removed.\n'

In [74]:
orders_master.to_csv(
    "../data/processed/olist_master_dataset.csv",
    index=False
)

print("=" * 90)
print("MASTER DATASET SAVED SUCCESSFULLY!")
print("=" * 90)

MASTER DATASET SAVED SUCCESSFULLY!
